<a href="https://colab.research.google.com/github/xyehya/documentation/blob/9.0/Unsloth-GRPO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%%capture
import os
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth vllm
else:
    # [NOTE] Do the below ONLY in Colab! Use [[pip install unsloth vllm]]
    !pip install --no-deps unsloth vllm
    !pip install matplotlib
    !pip install ipywidgets

In [ ]:
#@title Colab Extra Install { display-mode: "form" }
%%capture
import os
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth vllm
else:
    !pip install --no-deps unsloth vllm
    # [NOTE] Do the below ONLY in Colab! Use [[pip install unsloth vllm]]
    # Skip restarting message in Colab
    import sys, re, requests; modules = list(sys.modules.keys())
    for x in modules: sys.modules.pop(x) if "PIL" in x or "google" in x else None
    !pip install --no-deps bitsandbytes accelerate xformers==0.0.29.post3 peft "trl==0.15.2" triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf datasets huggingface_hub hf_transfer

    # vLLM requirements - vLLM breaks Colab due to reinstalling numpy
    f = requests.get("https://raw.githubusercontent.com/vllm-project/vllm/refs/heads/main/requirements/common.txt").content
    with open("vllm_requirements.txt", "wb") as file:
        file.write(re.sub(rb"(transformers|numpy|xformers)[^\n]{1,}\n", b"", f))
    !pip install -r vllm_requirements.txt

In [ ]:

from unsloth import FastLanguageModel
from trl import GRPOConfig, GRPOTrainer
import re
import os
import numpy as np
import torch
import matplotlib.pyplot as plt
from datasets import load_dataset
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, GenerationConfig, get_linear_schedule_with_warmup
from tqdm.notebook import tqdm  # Use notebook tqdm for better Jupyter integration


# Set configuration parameters
MODEL_PATH = "meta-llama/meta-Llama-3.1-8B-Instruct"
DATASET = "Jiayi-Pan/Countdown-Tasks-3to4"
USE_WANDB = False  # Set to False if you don't want to use wandb
GPU_MEMORY_UTIL = 0.85  # Adjust based on your RTX 4080's available memory
r=16 #Rank
max_seq_length=1750

# Define data loading functions
def make_dataset(dataset_name: str, max_samples=50000):
    """Load and prepare the dataset for training and testing."""
    print(f"Loading dataset {dataset_name}...")
    raw_dataset = (
        load_dataset(dataset_name, split="train").shuffle(seed=42).select(range(max_samples))
    )
    raw_dataset = raw_dataset.rename_column("target", "answer")
    raw_dataset = raw_dataset.rename_column("nums", "question")

    # Display a few examples
    print("\nSample data points:")
    for i in range(min(3, len(raw_dataset))):
        print(f"Example {i+1}:")
        print(f"  Question: {raw_dataset[i]['question']}")
        print(f"  Answer: {raw_dataset[i]['answer']}\n")

    # Split into train/test
    train_test_split = raw_dataset.train_test_split(test_size=0.1)
    train_dataset = train_test_split["train"]
    test_dataset = train_test_split["test"]
    print(f"Train set size: {len(train_dataset)}, Test set size: {len(test_dataset)}")

    return train_dataset, test_dataset

# Define custom dataset class
class CountdownDataset(Dataset):
    """Custom dataset for the Countdown task."""

    def __init__(self, dataset, tokenizer):
        self.dataset = dataset
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        item = self.dataset[idx]
        question = item["question"]
        answer = item["answer"]

        # Format the conversation for Llama models
        system_msg = "You are a helpful assistant. You first think about the reasoning process in your mind and then provide the user with the answe following the format requested by the user STRICTLY."
        user_msg = f"Using each number in this tensor only once {tuple(question)}, create an equation that equals {answer}. You can use basic arithmetic operations (+, -, *, /) and each number can only be used once. Show all of your work strictly between only ONE <think> </think> tags. And return the final equation and answer strictly between only ONE <answer> </answer> tags, for example <answer>(1 + 2) / 3</answer>. Do not skip tags"
        assistant_start = "Let me solve this step by step.\n<think>"

        # For Llama models, we can use the chat template
        conversation = [
            {"role": "system", "content": system_msg},
            {"role": "user", "content": user_msg},
            {"role": "assistant", "content": assistant_start}
        ]

        # Use the chat template if Llama model
        prompt = self.tokenizer.apply_chat_template(
            conversation,
            tokenize=False,
            add_generation_prompt=True
        )

        # Tokenize the prompt
        tokenized_prompt = self.tokenizer(
            prompt,
            return_tensors="pt",
            padding="max_length",
            max_length=1024,
            truncation=True,
        )

        return {
            "input_ids": tokenized_prompt.input_ids[0],
            "attention_mask": tokenized_prompt.attention_mask[0],
            "question": question,
            "answer": answer,
            "prompt": prompt,  # Include the raw text prompt
        }

# Cell 5: Define collate function

def custom_collate_fn(batch):
    """Custom collate function for the data loader."""
    input_ids = torch.stack([item["input_ids"] for item in batch])
    attention_mask = torch.stack([item["attention_mask"] for item in batch])
    questions = [item["question"] for item in batch]
    answers = [item["answer"] for item in batch]
    prompts = [item["prompt"] for item in batch]

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "question": questions,
        "answer": answers,
        "prompt": prompts,
    }

# Define reward functions

sample_counter = {"count": 0, "log_frequency": 1}

# Helper function to extract answers
def extract_answer_tag(text):
    """Extract content from <answer> tags."""
    answer_match = re.search(r"<answer>([\s\S]*?)</answer>", text)
    if answer_match:
        return answer_match.group(1).strip()
    return ""

def format_reward_func(completions, **kwargs) -> list[float]:
    """Reward function for format checking."""
    # Extract the content from completions
    contents = [completion[0]["content"] if isinstance(completion, list) else completion for completion in completions]

    rewards = []
    for i, content in enumerate(contents):
        # Check if we should log this sample
        should_log = (sample_counter["count"] % sample_counter["log_frequency"] == 0) and i == 0

        # Since the opening <think> tag is already in the prompt,
        # we only need to check for closing </think> and <answer> tags
        has_think_close = "</think>" in content
        answer_tags = re.findall(r"<answer>([\s\S]*?)</answer>", content)
        has_answer = len(answer_tags) == 1

        # Check if tags are in the correct order
        think_end = content.find("</think>")
        answer_start = content.find("<answer>")

        if has_think_close and has_answer and think_end < answer_start:
            rewards.append(1.0)
            if should_log:
                print(f"\n✅ FORMAT CHECK PASSED: Proper closing of </think> and use of <answer> tags")
        else:
            rewards.append(0.0)
            if should_log:
                issues = []
                if not has_think_close:
                    issues.append("Missing closing </think> tag")
                if not has_answer:
                    issues.append("Missing or multiple <answer> tags")
                if has_think_close and has_answer and think_end >= answer_start:
                    issues.append("Tags in wrong order")
                print(f"\n❌ FORMAT CHECK FAILED: {', '.join(issues)}")

    return rewards

def equation_reward_func(completions, question=None, answer=None, **kwargs) -> list[float]:
    """Reward function for equation correctness."""
    # Extract the content from completions
    contents = [completion[0]["content"] if isinstance(completion, list) else completion for completion in completions]

    # Ensure questions and answers are lists
    questions = question if question is not None else kwargs.get("questions", [])
    answers = answer if answer is not None else kwargs.get("answers", [])

    if not isinstance(questions, list):
        questions = [questions]
    if not isinstance(answers, list):
        answers = [answers]

    rewards = []
    for i, (content, answer_val, question_val) in enumerate(zip(contents, answers, questions)):
        # Check if we should log this sample
        should_log = (sample_counter["count"] % sample_counter["log_frequency"] == 0) and i == 0

        if should_log:
            sample_counter["count"] += 1
            print(f"\n{'='*60}")
            print(f"SAMPLE OUTPUT #{sample_counter['count']}")
            print(f"{'='*60}")
            print(f"QUESTION: Create an equation using {question_val} that equals {answer_val}")
            print(f"\nMODEL OUTPUT:\n{content}")

        try:
            # Extract the answer from the tags
            answer_tags = re.findall(r"<answer>([\s\S]*?)</answer>", content)

            # Check for exactly one answer tag
            if len(answer_tags) != 1:
                rewards.append(0.0)
                if should_log:
                    print(f"\n❌ EQUATION CHECK: Expected exactly one <answer> tag, found {len(answer_tags)}")
                continue

            # Get the equation from the answer tag
            equation = answer_tags[0].strip()
            used_numbers = [int(n) for n in re.findall(r"\d+", equation)]

            if sorted(used_numbers) != sorted(question_val):
                if should_log:
                    print(f"\n❌ EQUATION CHECK: Numbers mismatch - Used {used_numbers} vs Required {question_val}")
                rewards.append(0.0)
                continue

            allowed_pattern = r"^[\d+\-*/().\s]+$"
            if not re.match(allowed_pattern, equation):
                if should_log:
                    print(f"\n❌ EQUATION CHECK: Invalid format in equation '{equation}'")
                rewards.append(0.0)
                continue

            result = eval(equation, {"__builtins__": None}, {})

            if abs(float(result) - float(answer_val)) < 1e-5:
                rewards.append(1.0)
                if should_log:
                    print(f"\n✅ EQUATION CHECK: {equation} = {result} (Expected: {answer_val})")
            else:
                if should_log:
                    print(f"\n❌ EQUATION CHECK: {equation} = {result} (Expected: {answer_val})")
                rewards.append(0.0)
        except Exception as e:
            rewards.append(0.0)
            if should_log:
                print(f"\n❌ EQUATION ERROR: {str(e)}")

        if should_log:
            print(f"{'='*60}\n")

    return rewards


def log_example(completion, question, answer, format_reward, equation_reward):
    """Log an example with rewards for visualization."""
    print(f"\n{'=' * 50}")
    print(f"Question: Create an equation using {question} that equals {answer}")
    print(f"\nCompletion:\n{completion}")
    print(f"\nFormat reward: {format_reward}")
    print(f"Equation reward: {equation_reward}")
    print(f"Total reward: {format_reward + equation_reward}")
    print(f"{'=' * 50}\n")





🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
INFO 04-03 14:43:48 [__init__.py:239] Automatically detected platform cuda.


In [ ]:
#Load model and tokenizer
print("Loading model and tokenizer...")

# Load the model with Unsloth optimizations, 4-bit quantization, and vLLM for faster inference
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_PATH,
    max_seq_length=max_seq_length,
    dtype=torch.bfloat16,
    load_in_4bit=True,
    fast_inference=True,  # Enable vLLM for faster inference
    gpu_memory_utilization=GPU_MEMORY_UTIL,  # Control GPU memory usage
)

# Set pad token to eos token (important for batching)
tokenizer.pad_token = tokenizer.eos_token

# Apply LoRA fine-tuning with unsloth optimizations
model = FastLanguageModel.get_peft_model(
    model,
    r=r,
    lora_alpha=r,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    use_gradient_checkpointing="unsloth",  # Use unsloth's optimized checkpointing
    random_state=3407,  # For reproducibility
)

print("Model loaded successfully!")

# Cell 9: Prepare datasets
print("Preparing datasets...")
train_dataset, test_dataset = make_dataset(DATASET, max_samples=5000)  # Reduced for Jupyter demo

# Create custom datasets
train_custom_dataset = CountdownDataset(train_dataset, tokenizer)
test_custom_dataset = CountdownDataset(test_dataset, tokenizer)

# Create data loaders
train_dataloader = DataLoader(
    train_custom_dataset,
    batch_size=1,
    shuffle=True,
    collate_fn=custom_collate_fn,
)


Loading model and tokenizer...
==((====))==  Unsloth 2025.3.19: Fast Llama patching. Transformers: 4.50.2. vLLM: 0.8.2.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.557 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 8.0. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.29.post3. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: vLLM loading unsloth/meta-llama-3.1-8b-instruct-unsloth-bnb-4bit with actual GPU utilization = 84.03%
Unsloth: Your GPU has CUDA compute capability 8.0 with VRAM = 39.56 GB.
Unsloth: Using conservativeness = 1.0. Chunked prefill tokens = 1750. Num Sequences = 320.
Unsloth: vLLM's KV Cache can use up to 26.91 GB. Also swap space = 6 GB.
INFO 04-03 14:44:06 [config.py:585] This model supports multiple tasks: {'classify', 'score', 'reward', 'embed', 'generate'}. Defaulting to 

Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]


Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]


INFO 04-03 14:44:13 [punica_selector.py:18] Using PunicaWrapperGPU.
INFO 04-03 14:44:14 [model_runner.py:1146] Model loading took 5.9458 GB and 4.910246 seconds
INFO 04-03 14:44:17 [worker.py:267] Memory profiling takes 2.91 seconds
INFO 04-03 14:44:17 [worker.py:267] the current vLLM instance can use total_gpu_memory (39.56GiB) x gpu_memory_utilization (0.84) = 33.24GiB
INFO 04-03 14:44:17 [worker.py:267] model weights take 5.95GiB; non_torch_memory takes 0.09GiB; PyTorch activation peak memory takes 1.49GiB; the rest of the memory reserved for KV Cache is 25.71GiB.
INFO 04-03 14:44:17 [executor_base.py:111] # cuda blocks: 13163, # CPU blocks: 3072
INFO 04-03 14:44:17 [executor_base.py:116] Maximum concurrency for 1750 tokens per request: 120.35x
INFO 04-03 14:44:22 [model_runner.py:1442] Capturing cudagraphs for decoding. This may lead to unexpected consequences if the model is not static. To run the model in eager mode, set 'enforce_eager=True' or use '--enforce-eager' in the CLI. I

Capturing CUDA graph shapes: 100%|██████████| 43/43 [00:56<00:00,  1.31s/it]

INFO 04-03 14:45:18 [model_runner.py:1570] Graph capturing finished in 56 secs, took 0.86 GiB
INFO 04-03 14:45:18 [llm_engine.py:447] init engine (profile, create kv cache, warmup model) took 64.49 seconds



Unsloth 2025.3.19 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


Model loaded successfully!
Preparing datasets...
Loading dataset Jiayi-Pan/Countdown-Tasks-3to4...

Sample data points:
Example 1:
  Question: [95, 21, 3]
  Answer: 88

Example 2:
  Question: [72, 30, 29]
  Answer: 72

Example 3:
  Question: [67, 69, 69]
  Answer: 71

Train set size: 4500, Test set size: 500


In [ ]:
# Set up GRPO configuration
print("Setting up GRPO configuration...")
# Define max prompt length for better memory management
max_prompt_length = 50

# Configure the GRPO training with optimized settings
grpo_config = GRPOConfig(
    # Performance settings
    learning_rate=5e-6,
    adam_beta1=0.9,
    adam_beta2=0.99,
    weight_decay=0.1,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    optim="paged_adamw_8bit",  # Memory-efficient optimizer
    per_device_train_batch_size=8,
    gradient_accumulation_steps=1,  # Increase for smoother training
    num_generations=2,  # Reduced for Jupyter demo
    max_prompt_length=max_prompt_length,
    max_completion_length= max_seq_length - max_prompt_length,
    #num_train_epochs=1,
    max_steps=1000,  # Reduced for Jupyter demo
    # Logging and checkpointing
    logging_steps=1,
    save_steps=50,
    save_total_limit=2,

    # Stability settings
    max_grad_norm=0.1,

    # Additional settings
    label_names=[],  # Needed for custom reward function
    fp16=False,  # Using bf16 instead for RTX 4080
    bf16=True,
    # report_to="wandb" if USE_WANDB else "none",
    output_dir="./outputs",
)

# Create reward functions and GRPO trainer
print("Setting up GRPO trainer...")

# Create the GRPO trainer with direct reward functions
trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,  # Updated parameter name for compatibility
    args=grpo_config,
    train_dataset=train_custom_dataset,
    reward_funcs= [
        format_reward_func,
        equation_reward_func
    ]

)
print("GRPO trainer is ready!")

Setting up GRPO configuration...
Setting up GRPO trainer...
GRPO trainer is ready!


In [ ]:
# Start the trainer
print("Starting training...")
trainer.train()
# Use trainer.train(resume_from_checkpoint=True) when resuming a saved checkpoint
print("Training complete!")

Starting training...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 4,500 | Num Epochs = 1 | Total steps = 1,000
O^O/ \_/ \    Batch size per device = 8 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (8 x 1 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040/8,000,000,000 (0.52% trained)
wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.
wandb: Currently logged in as: ykar (ykar-deloitte) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin



❌ FORMAT CHECK FAILED: Missing or multiple <answer> tags

SAMPLE OUTPUT #1
QUESTION: Create an equation using [76, 86, 43] that equals 53

MODEL OUTPUT:
</think>

46 
76 / 86 = 0.887 
0.887 * 43 = 38.21 
53 - 38.21 = 14.79

❌ EQUATION CHECK: Expected exactly one <answer> tag, found 0


Step,Training Loss,reward,reward_std,completion_length,kl,rewards / format_reward_func,rewards / equation_reward_func
1,0.000000,0.125000,0.176777,490.875000,0.000000,0.125000,0.000000
2,0.000000,0.000000,0.000000,976.125000,0.000000,0.000000,0.000000
3,0.000000,0.375000,0.530330,367.750000,0.000783,0.250000,0.125000
4,0.000000,0.125000,0.176777,651.250000,0.000735,0.125000,0.000000
5,0.000000,0.125000,0.176777,473.875000,0.000629,0.125000,0.000000
6,0.000000,0.000000,0.000000,610.750000,0.000689,0.000000,0.000000



❌ FORMAT CHECK FAILED: Missing closing </think> tag, Missing or multiple <answer> tags

SAMPLE OUTPUT #2
QUESTION: Create an equation using [56, 86, 3] that equals 82

MODEL OUTPUT:
First, I will try different combinations of the numbers (56, 86, 3) to find an equation that equals 82.

One possible combination is to start with the numbers 56 and 3. Since 56 is close to 50, I can multiply it by something close to 1.7 to get 82.

56 * 1.470588 (which equals 82)

However, 1.470588 is not one of the given numbers. I will use this as a clue to double-check the combination. I will use the first two numbers 86 and 3.

86 - 3 = 83 which is not a clue, no 56 isn't multiplied by something out of given numbers that equals this.

However, if the equation is a combination of the first 2 number 86, 56

Then 86 - 4. This gets me a bit further but you can't find 4.

Let's look at the remaining combination 56, 3, and I have possibly underutilized the number 86. 
If the equation adds the remaining numb

KeyboardInterrupt: 